```diff
 WEEK_FROM = None                      # "YYYY-MM-DD"; None = last 5 sessions in data
 WEEK_TO = None
 REFIT_WEEKS = 3                       # trailing weeks used by refit_iso
+TRAIL_WEEKS = 4                       # slow AUC sensor: trailing window INCLUDING current week
```

```diff
 ALARM_METRICS = ["auc", "ll_gap"]     # these can raise ALERT
+SLOW_METRICS = ["auc_trail"]          # trailing window; consecutive dips overlap by
+                                      # TRAIL_WEEKS-1 weeks, so NO 2-week upgrade rule
 REPORT_METRICS = ["skill", "red_precision", "green_false_safe",
                   "red_frac", "base_rate"]
```

Baseline — add the trailing band (insert immediately before `base = {...}`):

```diff
     keys = ALARM_METRICS + REPORT_METRICS
     bands = {k: {"p5": float(np.nanpercentile(hist[k], 5)),
                  "p50": float(np.nanpercentile(hist[k], 50)),
                  "p95": float(np.nanpercentile(hist[k], 95)),
                  "min": float(np.nanmin(hist[k])),
                  "max": float(np.nanmax(hist[k]))} for k in keys}
+
+    # trailing-window AUC: pooled over TRAIL_WEEKS consecutive holdout weeks.
+    # sampling sigma ~ 1/sqrt(n), so ~half the weekly noise -> resolves slower drift.
+    wks = hist["wk"].tolist()
+    by_wk = {w: g for w, g in pred.groupby("wk")}
+    trail = []
+    for i in range(TRAIL_WEEKS - 1, len(wks)):
+        g = pd.concat([by_wk[w] for w in wks[i - TRAIL_WEEKS + 1:i + 1]])
+        yy = g["is_target"].to_numpy(np.float64)
+        if 0 < yy.sum() < len(yy):
+            trail.append(float(roc_auc_score(yy, g["p"].to_numpy())))
+    if len(trail) >= 3:
+        t = np.array(trail)
+        bands["auc_trail"] = {"p5": float(np.percentile(t, 5)),
+                              "p50": float(np.percentile(t, 50)),
+                              "p95": float(np.percentile(t, 95)),
+                              "min": float(t.min()), "max": float(t.max())}
+        print(f"auc_trail ({TRAIL_WEEKS}wk pooled, {len(t)} windows): "
+              f"min {t.min():.5f}  p5 {np.percentile(t, 5):.5f}  "
+              f"p50 {np.percentile(t, 50):.5f}   [weekly auc std {hist['auc'].std():.5f}"
+              f" vs trailing std {t.std():.5f}]")
+    else:
+        print(f"auc_trail: too few holdout weeks for a {TRAIL_WEEKS}wk band -- skipped")
 
     base = {"model": MODEL_PATH,
```

Verdict — slow metrics, no 2-week upgrade:

```diff
             else:
                 watch.append(k)
                 notes.append(f"{k}={v:.4f} < p5 {b['p5']:.4f}")
+    for k in SLOW_METRICS:                       # overlapping windows -> no upgrade rule
+        v = m.get(k, np.nan)
+        b = bands.get(k)
+        if not np.isfinite(v) or b is None:
+            continue
+        if v < b["min"]:
+            alert.append(k)
+            notes.append(f"{k}={v:.5f} BELOW baseline min {b['min']:.5f} (slow drift)")
+        elif v < b["p5"]:
+            watch.append(k)
+            notes.append(f"{k}={v:.5f} < p5 {b['p5']:.5f} (slow drift)")
     for k in REPORT_METRICS:
```

Diagnosis:

```diff
 def diagnose(m, base, alert, watch):
     bad = set(alert) | set(watch)
     auc_bad = "auc" in bad
+    slow_bad = "auc_trail" in bad
     cal_bad = "ll_gap" in bad
     if auc_bad and cal_bad:
         return "ranking AND calibration degraded -> investigate data first, then RETRAIN"
     if auc_bad:
         return "ranking degraded, calibration fine -> RETRAIN (booster no longer separates)"
+    if slow_bad:
+        s = (f"weekly auc normal but trailing-{TRAIL_WEEKS}wk auc is low -> SLOW ranking "
+             f"decay; schedule a retrain (not urgent)")
+        return s + " ; calibration also stale -> REFIT ISO now" if cal_bad else s
     if cal_bad:
```

Week run — score the trailing window (insert after the current-week metrics):

```diff
     y, p, p_cal, meta = score_window(fz, src, bundle, d_from, d_to)
     m = metrics(y, p, p_cal, base["green"], base["red"])
+
+    if "auc_trail" in base["bands"]:                                       # slow sensor
+        d_trail = str((pd.Timestamp(d_from)
+                       - pd.Timedelta(weeks=TRAIL_WEEKS - 1)).date())
+        yt, pt, _pct, _mt = score_window(fz, src, bundle, d_trail, d_to)
+        m["n_trail"] = int(len(yt))
+        m["auc_trail"] = (float(roc_auc_score(yt, pt))
+                          if 0 < yt.sum() < len(yt) else np.nan)
+        print(f"  trailing window {d_trail} .. {d_to}: {m['n_trail']} rows")
```

Print + log:

```diff
     print("\n---- metrics ----")
-    for k in ["n", "n_pos", "base_rate", "auc", "ll_gap", "skill",
+    for k in ["n", "n_pos", "base_rate", "auc", "auc_trail", "ll_gap", "skill",
               "red_frac", "red_precision", "green_false_safe"]:
```

```diff
     row = {k: m.get(k, np.nan) for k in
-           ["n", "n_pos", "base_rate", "auc", "ll_gap", "skill",
+           ["n", "n_pos", "n_trail", "base_rate", "auc", "auc_trail", "ll_gap", "skill",
             "red_frac", "red_precision", "green_false_safe"]}
```

Rerun `MODE="baseline"` once to add the new band before the next weekly run. The printed weekly-vs-trailing std comparison tells you directly how much resolution the slow sensor buys on your data — expect roughly half.

In [ ]:
"""
monitor_weekly.py -- weekly health check for the deployed hazard model.

WHAT THE 181-WEEK ROLLING STUDY CHANGED (this is why iteration 1 was wrong):
  - skill on a 1-week test has std 0.020, range 0.55-0.65, FLAT mean over 3.5
    years. That spread is sampling noise on ~1000 test positives, not regime
    change. So skill is REPORTED but NEVER alarmed on -- alarming on weekly
    skill is alarming on noise.
  - auc has std 0.0036 and is flat across every regime in the span. Relative to
    its spread it is ~5x more stable than skill, so a genuine ranking
    degradation shows up HERE first. auc is the PRIMARY alarm.
  - ll_gap sits persistently slightly negative (mean -0.0019): the iso fold is
    always a little stale. That is NORMAL, not an alarm. Only a sustained or
    unusually large negative excursion means "recalibrate".

Therefore this monitor DIAGNOSES rather than just alarms:

    auc low   + ll_gap normal  ->  ranking degraded      -> RETRAIN (expensive)
    auc normal+ ll_gap low     ->  calibration stale     -> REFIT ISO (cheap)
    both low                   ->  both
    neither                    ->  healthy; skill wiggle is noise

MODES
  "baseline"  : build the empirical bands from the deployed model's own holdout
                predictions, sliced by week. Run once per deployed model.
  "week"      : score a new week through the deployed model, compare to bands,
                print verdict + diagnosis, append one row to the log.
  "refit_iso" : the cheap fix. Keeps the frozen booster, refits ONLY the
                isotonic on the trailing REFIT_WEEKS, writes a NEW bundle
                (never overwrites). Use when the diagnosis says calibration.

RULES
  M.1  Bands are empirical, from this exact model's holdout weeks. No opinions.
  M.2  Data sanity runs BEFORE any model metric. Sanity FAIL aborts -- most
       "model degradation" in practice is a broken feed.
  M.3  Alarm ladder: below p5 -> WATCH; same metric below p5 two weeks running
       -> ALERT; below the baseline MINIMUM -> immediate ALERT.
  M.4  GREEN/RED lamp thresholds are frozen at baseline time (the model's own
       0.50 / 0.90 p_cal quantiles) and stored in the baseline file, so weekly
       lamp stats are measured against the lamp as actually deployed.
  M.5  One summary row per run appends to the log. Over time that log IS the
       regime record -- de-calibration episodes are dated there.
"""

import json
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score

from common import Featurizer, load_manifest
from stage_5 import build_X, build_meta, TAUS, TOD_BIN_MIN     # needs __main__ guard in stage_5

# ---------------------------------------------------------------- CONFIG
MODE = "baseline"                     # "baseline" | "week" | "refit_iso"

FRAME = 3
STAGE0_TAG = "mnq-TICK-9-12am-i4"
BODY_TAG = "raw"

MANIFEST_PATH = f"stage-0/manifest_{STAGE0_TAG}_{FRAME}s.json"
BARS_PATH = f"stage-0/bars_{STAGE0_TAG}_{FRAME}s.pqt"
EVENTS_PATH = f"stage-0/events_{STAGE0_TAG}_{FRAME}s.pqt"
MODEL_PATH = f"stage-5/model_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.joblib"
PRED_PATH = f"stage-5/pred_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.pqt"

BASELINE_PATH = f"monitor/baseline_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.json"
LOG_PATH = f"monitor/log_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.pqt"
ROLLING_CSV = f"rolling/{STAGE0_TAG}_{BODY_TAG}_{FRAME}s.csv"   # optional cross-check

WEEK_FROM = None                      # "YYYY-MM-DD"; None = last 5 sessions in data
WEEK_TO = None
REFIT_WEEKS = 3                       # trailing weeks used by refit_iso

EXPECTED_BARS = 3600                  # 09:00-12:00 at 3s
MIN_BARS = 3500                       # below this = broken session (i4 filter level)
THIN_POS = 400                        # test week with fewer positives = report only
PLOT = True
# ----------------------------------------------------------------

ALARM_METRICS = ["auc", "ll_gap"]     # these can raise ALERT
REPORT_METRICS = ["skill", "red_precision", "green_false_safe",
                  "red_frac", "base_rate"]


# ---------------------------------------------------------------- metrics
def ll(y, p):
    p = np.clip(np.asarray(p, np.float64), 1e-15, 1 - 1e-15)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def metrics(y, p, p_cal, green, red):
    y = np.asarray(y, np.float64)
    a = float(y.mean())
    m = {"n": int(len(y)), "n_pos": int(y.sum()), "base_rate": a}
    if not (0.0 < a < 1.0):
        return m
    ll_cal = ll(y, p_cal)
    m["ll_cal"] = ll_cal
    m["ll_const"] = ll(y, np.full_like(y, a))
    m["skill"] = 1.0 - ll_cal / m["ll_const"]
    m["ll_gap"] = ll(y, p) - ll_cal                    # <0 => iso stale for this week
    m["auc"] = float(roc_auc_score(y, p))
    r = np.asarray(p_cal) >= red
    g = np.asarray(p_cal) < green
    m["red_frac"] = float(r.mean())
    m["red_precision"] = float(y[r].mean()) if r.any() else np.nan
    m["green_false_safe"] = float(y[g].mean()) if g.any() else np.nan
    return m


def week_key(ts):
    iso = pd.DatetimeIndex(ts).isocalendar()
    return iso["year"].astype(str) + "-W" + iso["week"].astype(str).str.zfill(2)


# ---------------------------------------------------------------- baseline
def build_baseline():
    pred = pd.read_parquet(PRED_PATH)
    bundle = joblib.load(MODEL_PATH)
    test_from = str(bundle.get("train_end", "2025-12-31"))
    pred = pred[pred["timestamp"] > test_from]                             # holdout only
    green = float(pred["p_cal"].quantile(0.50))                            # M.4
    red = float(pred["p_cal"].quantile(0.90))

    pred = pred.assign(wk=week_key(pred["timestamp"]))
    rows = []
    for wk, g in pred.groupby("wk"):
        m = metrics(g["is_target"].to_numpy(), g["p"].to_numpy(),
                    g["p_cal"].to_numpy(), green, red)
        m["wk"] = wk
        m["start"] = str(g["timestamp"].min().date())
        rows.append(m)
    hist = pd.DataFrame(rows).sort_values("start").reset_index(drop=True)
    hist = hist[hist["n_pos"] >= THIN_POS]                                 # drop stubs

    keys = ALARM_METRICS + REPORT_METRICS
    bands = {k: {"p5": float(np.nanpercentile(hist[k], 5)),
                 "p50": float(np.nanpercentile(hist[k], 50)),
                 "p95": float(np.nanpercentile(hist[k], 95)),
                 "min": float(np.nanmin(hist[k])),
                 "max": float(np.nanmax(hist[k]))} for k in keys}

    base = {"model": MODEL_PATH,
            "model_mtime": str(pd.Timestamp(os.path.getmtime(MODEL_PATH), unit="s")),
            "tag": bundle.get("tag"), "valid_from": str(bundle.get("valid_from")),
            "train_end": str(bundle.get("train_end")),
            "green": green, "red": red,
            "n_weeks": int(len(hist)), "bands": bands}
    os.makedirs(os.path.dirname(BASELINE_PATH), exist_ok=True)
    with open(BASELINE_PATH, "w") as f:
        json.dump(base, f, indent=2)

    print(f"baseline from {len(hist)} holdout weeks -> {BASELINE_PATH}")
    print(f"lamp: GREEN < {green:.8g}   RED >= {red:.8g}")
    print(pd.DataFrame(bands).T[["min", "p5", "p50", "p95", "max"]].to_string())

    if os.path.exists(ROLLING_CSV):                       # width cross-check
        rw = pd.read_csv(ROLLING_CSV)
        print(f"\ncross-check vs {len(rw)} rolling weeks (freshly-retrained models):")
        for k in ["skill", "auc", "ll_gap"]:
            if k in rw:
                print(f"  {k:8s} holdout std {hist[k].std():.4f}   "
                      f"rolling std {rw[k].std():.4f}   "
                      f"rolling p5 {np.nanpercentile(rw[k], 5):.4f}")

    if PLOT and len(hist):
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                            subplot_titles=["auc (PRIMARY alarm)",
                                            "ll_gap (calibration; <0 normal-ish)",
                                            "skill (report only -- noisy)"])
        for i, k in enumerate(["auc", "ll_gap", "skill"]):
            fig.add_trace(go.Scatter(x=hist["start"], y=hist[k], mode="lines+markers",
                                     name=k, showlegend=False), row=i + 1, col=1)
            fig.add_hline(y=bands[k]["p5"], line_dash="dot", row=i + 1, col=1)
        fig.update_layout(height=760, width=1300, template="plotly_white",
                          title=f"baseline weeks -- {STAGE0_TAG} {FRAME}s")
        fig.show()
    return base, hist


# ---------------------------------------------------------------- scoring a window
def load_frame(manifest):
    lo = pd.Timestamp(manifest["session_start"]).time()
    hi = pd.Timestamp(manifest["session_end"]).time()
    src = pd.read_parquet(manifest["source_file"])
    src = src[(src["timestamp"].dt.time >= lo) & (src["timestamp"].dt.time < hi)]
    raw1 = pd.read_parquet(manifest["raw_file"])
    raw1 = raw1[(raw1["timestamp"].dt.time >= lo) & (raw1["timestamp"].dt.time < hi)]
    raw1 = raw1.rename(columns={"Open": "rawOpen", "High": "rawHigh",
                                "Low": "rawLow", "Last": "rawLast"})
    src = src.merge(raw1[["timestamp", "rawOpen", "rawHigh", "rawLow", "rawLast"]],
                    on="timestamp", how="left")
    assert src[["rawOpen", "rawLast"]].notna().all().all(), "raw OHLC gaps vs source"
    return src


def sanity(bars, d_from, d_to):                                            # M.2
    b = bars[(bars["date"].astype(str) >= d_from) & (bars["date"].astype(str) <= d_to)]
    ok = True
    sizes = b.groupby(b["date"].astype(str)).size()
    for d, n in sizes.items():
        if n < MIN_BARS:
            print(f"  SANITY FAIL {d}: {n} bars (< {MIN_BARS})")
            ok = False
        elif n != EXPECTED_BARS:
            print(f"  note {d}: {n} bars (expected {EXPECTED_BARS}) -- gaps/early close")
    if b["timestamp"].duplicated().any():
        print("  SANITY FAIL: duplicate timestamps")
        ok = False
    if len(sizes) == 0:
        print("  SANITY FAIL: no sessions in window")
        ok = False
    return ok, list(sizes.index)


def score_window(fz, src, bundle, d_from, d_to):
    X, _ = build_X(fz, src, d_from, d_to)
    meta = build_meta(fz, d_from, d_to)
    assert list(bundle["feature_names"]) == list(_), "feature contract mismatch"
    p = bundle["booster"].predict(X, num_iteration=bundle["booster"].best_iteration)
    p_cal = bundle["iso"].predict(p)
    return meta["is_target"].to_numpy().astype(np.int8), p, p_cal, meta


# ---------------------------------------------------------------- weekly run
def verdict(m, base, prev_watch):                                          # M.3
    bands = base["bands"]
    watch, alert, notes = [], [], []
    for k in ALARM_METRICS:
        v = m.get(k, np.nan)
        if not np.isfinite(v):
            continue
        b = bands[k]
        if v < b["min"]:
            alert.append(k)
            notes.append(f"{k}={v:.4f} BELOW baseline min {b['min']:.4f}")
        elif v < b["p5"]:
            if k in prev_watch:
                alert.append(k)
                notes.append(f"{k}={v:.4f} < p5 {b['p5']:.4f} (2nd week running)")
            else:
                watch.append(k)
                notes.append(f"{k}={v:.4f} < p5 {b['p5']:.4f}")
    for k in REPORT_METRICS:
        v = m.get(k, np.nan)
        if np.isfinite(v):
            b = bands[k]
            if v < b["p5"] or v > b["p95"]:
                notes.append(f"(report) {k}={v:.4f} outside [{b['p5']:.4f}, {b['p95']:.4f}]")
    v = "ALERT" if alert else ("WATCH" if watch else "OK")
    return v, watch, alert, notes


def diagnose(m, base, alert, watch):
    bad = set(alert) | set(watch)
    auc_bad = "auc" in bad
    cal_bad = "ll_gap" in bad
    if auc_bad and cal_bad:
        return "ranking AND calibration degraded -> investigate data first, then RETRAIN"
    if auc_bad:
        return "ranking degraded, calibration fine -> RETRAIN (booster no longer separates)"
    if cal_bad:
        return "ranking fine, calibration stale -> REFIT ISO (cheap; MODE='refit_iso')"
    return "healthy -- weekly skill wiggle of +/-0.02 is sampling noise, not signal"


def run_week():
    with open(BASELINE_PATH) as f:
        base = json.load(f)
    bundle = joblib.load(MODEL_PATH)
    manifest = load_manifest(MANIFEST_PATH, TOD_BIN_MIN)
    bars = pd.read_parquet(BARS_PATH)
    events = pd.read_parquet(EVENTS_PATH)

    days = sorted(bars["date"].astype(str).unique())
    d_from = WEEK_FROM or days[-5]
    d_to = WEEK_TO or days[-1]
    print(f"window {d_from} .. {d_to}")

    print("data sanity:")
    ok, sess = sanity(bars, d_from, d_to)
    if not ok:
        print("VERDICT: DATA FAIL -- fix data prep and rerun. No model metrics read.")
        return None
    print(f"  {len(sess)} sessions, all >= {MIN_BARS} bars, no dupes")

    src = load_frame(manifest)
    fz = Featurizer(bars, events, manifest, TOD_BIN_MIN, TAUS)
    y, p, p_cal, meta = score_window(fz, src, bundle, d_from, d_to)
    m = metrics(y, p, p_cal, base["green"], base["red"])

    prev_watch = []
    if os.path.exists(LOG_PATH):
        log = pd.read_parquet(LOG_PATH)
        if len(log):
            prev_watch = str(log.iloc[-1].get("watch", "")).split("|")
            prev_watch = [w for w in prev_watch if w]
    v, watch, alert, notes = verdict(m, base, prev_watch)
    dx = diagnose(m, base, alert, watch)

    print("\n---- metrics ----")
    for k in ["n", "n_pos", "base_rate", "auc", "ll_gap", "skill",
              "red_frac", "red_precision", "green_false_safe"]:
        val = m.get(k, np.nan)
        b = base["bands"].get(k)
        ref = f"   [p5 {b['p5']:.4f} p50 {b['p50']:.4f} p95 {b['p95']:.4f}]" if b else ""
        print(f"  {k:18s} {val:12.6f}{ref}" if isinstance(val, float)
              else f"  {k:18s} {val:12d}")
    if m.get("n_pos", 0) < THIN_POS:
        print(f"  NOTE: thin week ({m['n_pos']} positives) -- metrics noisy, "
              f"treat alarms as WATCH only")

    print(f"\nVERDICT: {v}")
    for s in notes:
        print("  -", s)
    print(f"DIAGNOSIS: {dx}")

    row = {k: m.get(k, np.nan) for k in
           ["n", "n_pos", "base_rate", "auc", "ll_gap", "skill",
            "red_frac", "red_precision", "green_false_safe"]}
    row.update({"from": d_from, "to": d_to, "verdict": v,
                "watch": "|".join(watch), "alert": "|".join(alert),
                "diagnosis": dx, "run_at": str(pd.Timestamp.now())})
    os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)
    if os.path.exists(LOG_PATH):
        log = pd.concat([pd.read_parquet(LOG_PATH), pd.DataFrame([row])],
                        ignore_index=True)
    else:
        log = pd.DataFrame([row])
    log.to_parquet(LOG_PATH, index=False)                                  # M.5
    print(f"\nlogged -> {LOG_PATH}  ({len(log)} runs)")
    return m


# ---------------------------------------------------------------- cheap fix
def refit_iso():
    """Freeze the booster, refit ONLY the isotonic on the trailing REFIT_WEEKS.
    Writes a NEW bundle; never overwrites the deployed one."""
    bundle = joblib.load(MODEL_PATH)
    manifest = load_manifest(MANIFEST_PATH, TOD_BIN_MIN)
    bars = pd.read_parquet(BARS_PATH)
    events = pd.read_parquet(EVENTS_PATH)
    src = load_frame(manifest)

    days = sorted(bars["date"].astype(str).unique())
    d_to = days[-1]
    d_from = str((pd.Timestamp(d_to) - pd.Timedelta(weeks=REFIT_WEEKS)).date())
    print(f"refit iso on {d_from} .. {d_to}")

    fz = Featurizer(bars, events, manifest, TOD_BIN_MIN, TAUS)
    X, names = build_X(fz, src, d_from, d_to)
    meta = build_meta(fz, d_from, d_to)
    y = meta["is_target"].to_numpy().astype(np.int8)
    p = bundle["booster"].predict(X, num_iteration=bundle["booster"].best_iteration)

    old = bundle["iso"].predict(p)
    new_iso = IsotonicRegression(out_of_bounds="clip").fit(p, y)
    new = new_iso.predict(p)
    print(f"  rows {len(y)}  positives {int(y.sum())}  base_rate {y.mean():.5f}")
    print(f"  ll old iso {ll(y, old):.6f}   ll new iso {ll(y, new):.6f}   "
          f"(in-sample for the new one -- expect it lower)")

    stamp = pd.Timestamp.now().strftime("%Y%m%d")
    out = MODEL_PATH.replace(".joblib", f"_iso{stamp}.joblib")
    nb = dict(bundle)
    nb["iso"] = new_iso
    nb["iso_refit_from"] = d_from
    nb["iso_refit_to"] = d_to
    joblib.dump(nb, out)
    print(f"  wrote {out}")
    print("  NEXT: point the worker's tag/path at it, rerun MODE='baseline' to "
          "rebuild bands for the new calibration, then resume weekly runs.")
    return out


# ---------------------------------------------------------------- usage
if __name__ == "__main__":
    if MODE == "baseline":
        base, hist = build_baseline()
    elif MODE == "week":
        m = run_week()
    elif MODE == "refit_iso":
        path = refit_iso()